# 🎸 Proyek Data Warehouse: Bandcamp Sales Analytics
Proyek ini membangun sebuah **Data Warehouse** berbasis *Star Schema* (model Kimball) untuk menganalisis penjualan musik independen di platform Bandcamp.

### **Prinsip Desain (Best Practice):**
- **Grain fakta yang eksplisit**: satu baris `Fact_Sales` = satu baris transaksi penjualan (`Order_ID` sebagai *degenerate dimension*).
- **Surrogate key**: setiap dimensi memakai *integer surrogate key* (SK) yang di-generate warehouse, terpisah dari *natural key* sumber.
- **Skema yang dideklarasikan**: DDL eksplisit dengan **PRIMARY KEY, FOREIGN KEY, dan INDEX** — bukan tipe hasil inferensi pandas.
- **Referential integrity**: setiap dimensi punya baris **"Unknown" (SK = -1)** sehingga fakta dengan atribut hilang tetap dapat di-join.
- **Conformed measure**: seluruh nilai uang dinormalisasi ke **USD** dan dibulatkan 2 desimal.

### **Alur Kerja (Pipeline):**
1. **Extract & Validate** — baca data mentah + *Data Quality Check*.
2. **Transform** — bentuk tabel Dimensi (Waktu, Lokasi, Artis, Item, Penggemar) ber-*surrogate key* + tabel Fakta.
3. **Load** — buat skema ber-DDL (PK/FK/Index) lalu muat data ke SQLite.
4. **OLAP Analysis** — 7 query analitik untuk menjawab pertanyaan bisnis dan ekspor hasil ke CSV.

> ⚠️ **Catatan integritas data:** kolom `Primary_Genre`, `Dim_Penggemar` (User), dan `Fan_Status` adalah **data sintetis** (tidak tersedia di sumber Bandcamp) dan ditandai eksplisit. Query 2 & 7 yang memakainya bersifat demonstrasi metodologi, bukan *insight* bisnis nyata.

In [10]:
import pandas as pd
import sqlite3
import numpy as np
import os
import warnings

# Konfigurasi Tampilan & Penyiapan Folder Output
warnings.filterwarnings('ignore')
os.makedirs('database', exist_ok=True)
os.makedirs('etl', exist_ok=True)
os.makedirs('olap', exist_ok=True)

# Path Dataset ('dataset/sample_bandcamp_sales.csv' untuk uji cepat)
RAW_DATA_PATH = 'dataset/sample_bandcamp_sales.csv'

print("Library berhasil di-import dan environment telah siap.")

Library berhasil di-import dan environment telah siap.


### **Tahap 1: Extract & Validation**
Pada tahap ini, kita memuat data dari file CSV mentah. Sebelum diproses lebih lanjut, data harus melewati filter **Data Quality Check** untuk memastikan tidak ada data kotor yang masuk ke Data Warehouse.

**Anomali yang ditangani:**
* Menghapus duplikasi transaksi berdasarkan ID Unik (`_id`).
* Menghapus baris jika nilai pada kolom krusial kosong (NULL).
* Menghapus data anomali di mana nominal pendapatan bernilai negatif.
* Menstandardisasi tipe item agar konsisten.

In [11]:
df_raw = pd.read_csv(RAW_DATA_PATH)

def validate_data(df):
    initial_count = len(df)

    for col in ['amount_paid_usd', 'amount_paid', 'item_price']:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    # Hapus duplikasi transaksi berdasarkan natural key (_id)
    df = df.drop_duplicates(subset=['_id'], keep='first')

    # Hapus NULL pada kolom krusial (kolom opsional seperti album_title dibiarkan)
    df = df.dropna(subset=['_id', 'artist_name', 'amount_paid_usd', 'item_type'])

    # Hapus data anomali: pendapatan negatif
    df = df[df['amount_paid_usd'] >= 0]

    # Standardisasi item_type agar konsisten
    valid_types = ['a', 't', 'p']
    df.loc[~df['item_type'].isin(valid_types), 'item_type'] = 'other'

    print(f"Audit: {initial_count - len(df)} baris anomali berhasil dihapus.")
    print(f"Data Bersih: {len(df)} baris siap diproses.")
    return df.reset_index(drop=True)

df_clean = validate_data(df_raw)

Audit: 0 baris anomali berhasil dihapus.
Data Bersih: 1000 baris siap diproses.


### **Tahap 2: Transformasi — Tabel Dimensi (Surrogate Keys)**
Data bersih dipecah menjadi tabel Dimensi. Mengikuti *best practice* Kimball:

- Setiap dimensi memakai **integer surrogate key** (`*_SK`) yang di-generate warehouse (1, 2, 3, …), terpisah dari *natural/business key* sumber yang tetap disimpan untuk *lineage*.
- Setiap dimensi diberi **baris "Unknown" (`SK = -1`)** untuk menampung fakta dengan atribut hilang tanpa merusak join.
- **Dim_Waktu** dibangun sebagai *date dimension* penuh (rentang tanggal kontigu) dengan atribut kalender lengkap (hari, nama bulan, kuartal, akhir pekan).
- **Dim_Item.Harga_Satuan_USD** dinormalisasi ke USD (memakai kurs implisit `amount_paid_usd / amount_paid`) agar tidak mencampur mata uang.

In [12]:
#  Helper: baris "Unknown" (surrogate key = -1)
UNKNOWN_SK = -1

def add_unknown_row(dim, sk_col, unknown_values):
    """Menambahkan 1 baris 'Unknown' (SK = -1) di awal sebuah dimensi."""
    unknown = {sk_col: UNKNOWN_SK, **unknown_values}
    return pd.concat([pd.DataFrame([unknown]), dim], ignore_index=True)

#  Dim_Waktu  — date dimension penuh (date spine kontigu)
df_clean['datetime'] = pd.to_datetime(df_clean['utc_date'], unit='s')
df_clean['Time_ID']  = df_clean['datetime'].dt.strftime('%Y%m%d').astype(int)

date_min = df_clean['datetime'].dt.normalize().min()
date_max = df_clean['datetime'].dt.normalize().max()
spine = pd.date_range(date_min, date_max, freq='D')

dim_waktu = pd.DataFrame({
    'Time_ID':    spine.strftime('%Y%m%d').astype(int),
    'Tanggal':    spine.date.astype(str),
    'Hari':       spine.day_name(),
    'Bulan':      spine.month,
    'Nama_Bulan': spine.month_name(),
    'Kuartal':    spine.quarter,
    'Tahun':      spine.year,
    'Is_Weekend': spine.dayofweek.isin([5, 6]),
    # Bandcamp Friday = Jumat pertama tiap bulan (aproksimasi kebijakan resmi)
    'Bandcamp_Friday': (spine.dayofweek == 4) & (spine.day <= 7),
})
dim_waktu = add_unknown_row(dim_waktu, 'Time_ID', {
    'Tanggal': None, 'Hari': 'Unknown', 'Bulan': -1, 'Nama_Bulan': 'Unknown',
    'Kuartal': -1, 'Tahun': -1, 'Is_Weekend': False, 'Bandcamp_Friday': False,
})

#  Dim_Artis  (natural key = Nama_Artis)
dim_artis = pd.DataFrame({'Nama_Artis': sorted(df_clean['artist_name'].dropna().unique())})
dim_artis.insert(0, 'Artis_SK', range(1, len(dim_artis) + 1))
dim_artis['Primary_Genre'] = '(synthetic) Indie / Alternative'  # data sintetis — bukan dari sumber
dim_artis = add_unknown_row(dim_artis, 'Artis_SK',
                            {'Nama_Artis': 'Unknown', 'Primary_Genre': 'Unknown'})

#  Dim_Lokasi  (natural key = Kode_Negara / country_code)
dim_lokasi = (df_clean[['country_code', 'country']]
              .dropna(subset=['country_code'])
              .drop_duplicates(subset=['country_code'])
              .sort_values('country_code').reset_index(drop=True))
dim_lokasi.columns = ['Kode_Negara', 'Negara']
dim_lokasi.insert(0, 'Lokasi_SK', range(1, len(dim_lokasi) + 1))
dim_lokasi = add_unknown_row(dim_lokasi, 'Lokasi_SK',
                             {'Kode_Negara': '??', 'Negara': 'Unknown'})

#  Dim_Item  (natural key = Nama_Item / item_description)
#     Harga_Satuan_USD dinormalisasi ke USD (kurs implisit)
rate = df_clean['amount_paid_usd'] / df_clean['amount_paid'].where(df_clean['amount_paid'] > 0)
df_clean['item_price_usd'] = (df_clean['item_price'] * rate).round(2)
df_clean['item_price_usd'] = df_clean['item_price_usd'].fillna(df_clean['item_price'].round(2))

type_mapping = {'a': 'Digital Album', 't': 'Digital Track', 'p': 'Physical / Merch', 'other': 'Other'}
df_clean['Tipe_Item'] = df_clean['item_type'].map(type_mapping)

dim_item = (df_clean.dropna(subset=['item_description'])
            .groupby('item_description', as_index=False)
            .agg(Tipe_Item=('Tipe_Item', 'first'),
                 Harga_Satuan_USD=('item_price_usd', 'median')))
dim_item['Harga_Satuan_USD'] = dim_item['Harga_Satuan_USD'].round(2)
dim_item.rename(columns={'item_description': 'Nama_Item'}, inplace=True)
dim_item = dim_item.sort_values('Nama_Item').reset_index(drop=True)
dim_item.insert(0, 'Item_SK', range(1, len(dim_item) + 1))
dim_item = dim_item[['Item_SK', 'Nama_Item', 'Tipe_Item', 'Harga_Satuan_USD']]
dim_item = add_unknown_row(dim_item, 'Item_SK',
                           {'Nama_Item': 'Unknown', 'Tipe_Item': 'Unknown', 'Harga_Satuan_USD': 0.0})

#  Dim_Penggemar  (SINTETIS — ditandai eksplisit)
np.random.seed(42)
df_clean['_user_natural'] = "USR-" + np.random.randint(1, 500, df_clean.shape[0]).astype(str)
dim_penggemar = pd.DataFrame({'User_Natural_Key': sorted(df_clean['_user_natural'].unique())})
dim_penggemar.insert(0, 'Penggemar_SK', range(1, len(dim_penggemar) + 1))
dim_penggemar['Nama_User'] = "Anonymous Fan " + dim_penggemar['User_Natural_Key']
dim_penggemar['Fan_Status'] = np.where(
    np.random.rand(len(dim_penggemar)) > 0.8, 'Subscriber', 'Standard')  # sintetis
dim_penggemar = add_unknown_row(dim_penggemar, 'Penggemar_SK',
                                {'User_Natural_Key': 'UNKNOWN', 'Nama_User': 'Unknown',
                                 'Fan_Status': 'Unknown'})

print("5 Tabel Dimensi (ber-surrogate key + baris Unknown) berhasil dibentuk.")
print(f"   Dim_Waktu: {len(dim_waktu)} baris  |  Dim_Artis: {len(dim_artis)}  |  "
      f"Dim_Lokasi: {len(dim_lokasi)}  |  Dim_Item: {len(dim_item)}  |  "
      f"Dim_Penggemar: {len(dim_penggemar)}")

5 Tabel Dimensi (ber-surrogate key + baris Unknown) berhasil dibentuk.
   Dim_Waktu: 2 baris  |  Dim_Artis: 761  |  Dim_Lokasi: 36  |  Dim_Item: 872  |  Dim_Penggemar: 423


### **Tahap 3: Membangun Tabel Fakta (Fact_Sales)**
Tabel Fakta adalah pusat *Star Schema*. **Grain**-nya eksplisit: *satu baris = satu transaksi penjualan*.

- *Natural key* tiap baris (`artist_name`, `country_code`, `item_description`, user) diterjemahkan menjadi **foreign key surrogate** (`Artis_SK`, `Lokasi_SK`, `Item_SK`, `Penggemar_SK`) lewat join ke dimensi; yang tak cocok jatuh ke **Unknown (-1)**.
- `Order_ID` disimpan sebagai *degenerate dimension* (identitas transaksi tanpa tabel dimensi tersendiri); `Sale_SK` menjadi surrogate PK fakta.
- *Measures*: `Gross_Rev`, `Bandcamp_Cut` (Digital 15% / Physical 10% / Bandcamp Friday 0%), dan `Artist_Revenue` — semua dibulatkan 2 desimal. Flag Bandcamp Friday diambil via **lookup map** (memperbaiki bug *positional merge* versi sebelumnya).

In [13]:
fact = df_clean.copy()
fact['Quantity']  = 1
fact['Gross_Rev'] = fact['amount_paid_usd'].round(2)

# --- Kalkulasi revenue split ---
# FIX BUG: flag Bandcamp Friday diambil via lookup map dari Dim_Waktu (by Time_ID),
# bukan via merge posisional yang rawan salah-align.
bcf_map = dim_waktu.set_index('Time_ID')['Bandcamp_Friday'].to_dict()
fact['_is_bcf'] = fact['Time_ID'].map(bcf_map).fillna(False).astype(bool)

# Potongan Bandcamp: Digital 15%, Physical 10%, Bandcamp Friday 0%
fact['Cut_Rate']       = np.where(fact['item_type'] == 'p', 0.10, 0.15)
fact['Cut_Rate']       = np.where(fact['_is_bcf'], 0.0, fact['Cut_Rate'])
fact['Bandcamp_Cut']   = (fact['Gross_Rev'] * fact['Cut_Rate']).round(2)
fact['Artist_Revenue'] = (fact['Gross_Rev'] - fact['Bandcamp_Cut']).round(2)

# --- Resolusi natural key -> surrogate key ---
# PENTING: join hanya ke baris dimensi REAL (SK != -1). Baris 'Unknown' sengaja
# dikecualikan agar data asli yang kebetulan bernilai 'Unknown'/'??' tidak
# double-match (fan-out). Yang benar-benar tak cocok baru jatuh ke -1 via fillna.
real_artis = dim_artis[dim_artis['Artis_SK'] != UNKNOWN_SK]
real_lokasi = dim_lokasi[dim_lokasi['Lokasi_SK'] != UNKNOWN_SK]
real_item = dim_item[dim_item['Item_SK'] != UNKNOWN_SK]
real_peng = dim_penggemar[dim_penggemar['Penggemar_SK'] != UNKNOWN_SK]

fact = fact.merge(real_artis[['Artis_SK', 'Nama_Artis']],
                  left_on='artist_name', right_on='Nama_Artis', how='left')
fact = fact.merge(real_lokasi[['Lokasi_SK', 'Kode_Negara']],
                  left_on='country_code', right_on='Kode_Negara', how='left')
fact = fact.merge(real_item[['Item_SK', 'Nama_Item']],
                  left_on='item_description', right_on='Nama_Item', how='left')
fact = fact.merge(real_peng[['Penggemar_SK', 'User_Natural_Key']],
                  left_on='_user_natural', right_on='User_Natural_Key', how='left')

for sk in ['Artis_SK', 'Lokasi_SK', 'Item_SK', 'Penggemar_SK']:
    fact[sk] = fact[sk].fillna(UNKNOWN_SK).astype(int)
# Time_ID selalu ada di spine, tapi tetap dijaga agar FK valid
fact['Time_ID'] = fact['Time_ID'].where(fact['Time_ID'].isin(dim_waktu['Time_ID']), UNKNOWN_SK)

# --- Bentuk Tabel Fakta ---
# Order_ID = degenerate dimension (natural key transaksi). Sale_SK = surrogate PK fakta.
fact_sales = fact[['_id', 'Time_ID', 'Artis_SK', 'Lokasi_SK', 'Item_SK', 'Penggemar_SK',
                   'Quantity', 'Gross_Rev', 'Bandcamp_Cut', 'Artist_Revenue']].copy()
fact_sales.rename(columns={'_id': 'Order_ID'}, inplace=True)
fact_sales.insert(0, 'Sale_SK', range(1, len(fact_sales) + 1))

assert len(fact_sales) == len(df_clean), "Fact row count harus sama dengan data bersih (tidak boleh fan-out)!"
print("✅ Tabel Fakta (Fact_Sales) berhasil dihitung. Grain = 1 baris per transaksi.")
display(fact_sales.head(3))

✅ Tabel Fakta (Fact_Sales) berhasil dihitung. Grain = 1 baris per transaksi.


,Sale_SK,Order_ID,Time_ID,Artis_SK,Lokasi_SK,Item_SK,Penggemar_SK,Quantity,Gross_Rev,Bandcamp_Cut,Artist_Revenue
0,1,1599688803.5175&//girlbanddublin.bandcamp.com/...,20200909,266,16,417,5,1,9.99,1.50,8.49
1,2,1599688805.27838&//maharettarecords.bandcamp.c...,20200909,338,14,497,321,1,1.30,0.20,1.10
2,3,1599688805.90646&//maharettarecords.bandcamp.c...,20200909,160,14,716,243,1,3.90,0.58,3.32


### **Tahap 4a: Load ke SQLite (Skema ber-DDL)**
Berbeda dari sekadar `to_sql`, di sini skema dibuat lewat **DDL eksplisit**:

- **PRIMARY KEY** pada setiap surrogate key dimensi & `Sale_SK` fakta.
- **FOREIGN KEY** dari `Fact_Sales` ke seluruh dimensi (referential integrity).
- **INDEX** pada kolom FK fakta untuk mempercepat join OLAP.
- `PRAGMA foreign_keys = ON` + `PRAGMA foreign_key_check` untuk memvalidasi tidak ada fakta yatim.

Urutan muat: **dimensi dulu, fakta belakangan** (agar FK selalu valid). Proses bersifat *idempotent* (drop-create).

In [14]:
conn = sqlite3.connect('database/bandcamp_dw.db')
conn.execute("PRAGMA foreign_keys = ON;")
cur = conn.cursor()

# Idempotent: drop tabel lama (fakta dulu, baru dimensi)
for t in ['Fact_Sales', 'Dim_Waktu', 'Dim_Lokasi', 'Dim_Artis', 'Dim_Item', 'Dim_Penggemar']:
    cur.execute(f"DROP TABLE IF EXISTS {t};")

# --- DDL eksplisit: PK, FK, tipe kolom ---
cur.executescript("""
CREATE TABLE Dim_Waktu (
    Time_ID         INTEGER PRIMARY KEY,
    Tanggal         TEXT,
    Hari            TEXT,
    Bulan           INTEGER,
    Nama_Bulan      TEXT,
    Kuartal         INTEGER,
    Tahun           INTEGER,
    Is_Weekend      INTEGER,
    Bandcamp_Friday INTEGER
);
CREATE TABLE Dim_Lokasi (
    Lokasi_SK   INTEGER PRIMARY KEY,
    Kode_Negara TEXT,
    Negara      TEXT
);
CREATE TABLE Dim_Artis (
    Artis_SK      INTEGER PRIMARY KEY,
    Nama_Artis    TEXT,
    Primary_Genre TEXT
);
CREATE TABLE Dim_Item (
    Item_SK          INTEGER PRIMARY KEY,
    Nama_Item        TEXT,
    Tipe_Item        TEXT,
    Harga_Satuan_USD REAL
);
CREATE TABLE Dim_Penggemar (
    Penggemar_SK     INTEGER PRIMARY KEY,
    User_Natural_Key TEXT,
    Nama_User        TEXT,
    Fan_Status       TEXT
);
CREATE TABLE Fact_Sales (
    Sale_SK        INTEGER PRIMARY KEY,
    Order_ID       TEXT,
    Time_ID        INTEGER,
    Artis_SK       INTEGER,
    Lokasi_SK      INTEGER,
    Item_SK        INTEGER,
    Penggemar_SK   INTEGER,
    Quantity       INTEGER,
    Gross_Rev      REAL,
    Bandcamp_Cut   REAL,
    Artist_Revenue REAL,
    FOREIGN KEY (Time_ID)      REFERENCES Dim_Waktu(Time_ID),
    FOREIGN KEY (Artis_SK)     REFERENCES Dim_Artis(Artis_SK),
    FOREIGN KEY (Lokasi_SK)    REFERENCES Dim_Lokasi(Lokasi_SK),
    FOREIGN KEY (Item_SK)      REFERENCES Dim_Item(Item_SK),
    FOREIGN KEY (Penggemar_SK) REFERENCES Dim_Penggemar(Penggemar_SK)
);
""")

# Muat dimensi dulu, lalu fakta (append ke skema yang sudah dibuat)
dim_waktu.to_sql('Dim_Waktu', conn, if_exists='append', index=False)
dim_lokasi[['Lokasi_SK', 'Kode_Negara', 'Negara']].to_sql('Dim_Lokasi', conn, if_exists='append', index=False)
dim_artis[['Artis_SK', 'Nama_Artis', 'Primary_Genre']].to_sql('Dim_Artis', conn, if_exists='append', index=False)
dim_item.to_sql('Dim_Item', conn, if_exists='append', index=False)
dim_penggemar[['Penggemar_SK', 'User_Natural_Key', 'Nama_User', 'Fan_Status']].to_sql('Dim_Penggemar', conn, if_exists='append', index=False)
fact_sales.to_sql('Fact_Sales', conn, if_exists='append', index=False)

# Index pada kolom FK fakta (akselerasi join OLAP pada dataset besar)
for col in ['Time_ID', 'Artis_SK', 'Lokasi_SK', 'Item_SK', 'Penggemar_SK']:
    cur.execute(f"CREATE INDEX idx_fact_{col} ON Fact_Sales({col});")

# Validasi referential integrity
violations = cur.execute("PRAGMA foreign_key_check;").fetchall()
conn.commit()
conn.close()

print("✅ Tahap 4a Selesai: skema DWH dibuat dengan PK/FK/Index di database/bandcamp_dw.db")
print(f"   Pelanggaran Foreign Key terdeteksi: {len(violations)} (harus 0)")

✅ Tahap 4a Selesai: skema DWH dibuat dengan PK/FK/Index di database/bandcamp_dw.db
   Pelanggaran Foreign Key terdeteksi: 0 (harus 0)


### **Tahap 4b: Ekspor Data Warehouse ke CSV**
Setelah data tersimpan di *database*, kita mengekstrak tabel-tabel tersebut ke dalam bentuk CSV di folder `etl/`. Format CSV ini sangat fleksibel dan mudah diimpor ke berbagai platform Business Intelligence (seperti Google Looker Studio atau Tableau).

In [15]:
conn = sqlite3.connect('database/bandcamp_dw.db')

tables = ['Fact_Sales', 'Dim_Item', 'Dim_Waktu', 'Dim_Penggemar', 'Dim_Artis', 'Dim_Lokasi']

for table in tables:
    df_export = pd.read_sql_query(f"SELECT * FROM {table}", conn)
    df_export.to_csv(f'etl/export_{table}.csv', index=False)

conn.close()
print("✅ Tahap 4b Selesai: Seluruh tabel DWH berhasil diekspor ke folder etl/")

✅ Tahap 4b Selesai: Seluruh tabel DWH berhasil diekspor ke folder etl/


### **Tahap 5: Eksekusi Advanced OLAP Queries**
Tahap terakhir adalah mengolah metrik lanjutan untuk menjawab 7 *Business Questions*. Kita akan menggunakan SQL untuk menghitung Average Transaction Value (ATV), Average Revenue Per User (ARPU), dan membandingkan rata-rata pendapatan harian agar *insight* yang dihasilkan adil dan akurat.

In [16]:
conn = sqlite3.connect('database/bandcamp_dw.db')

queries = {
    "1_revenue_bulanan": """
        SELECT w.Tahun, w.Bulan,
               COUNT(f.Order_ID) as Total_Transaksi,
               ROUND(SUM(f.Gross_Rev), 2) as Total_Gross_Revenue,
               ROUND(SUM(f.Artist_Revenue), 2) as Total_Artist_Revenue,
               ROUND(SUM(f.Gross_Rev) / COUNT(f.Order_ID), 2) as Avg_Transaction_Value
        FROM Fact_Sales f JOIN Dim_Waktu w ON f.Time_ID = w.Time_ID
        GROUP BY w.Tahun, w.Bulan ORDER BY w.Tahun, w.Bulan;
    """,
    # CATATAN: Primary_Genre = data SINTETIS, query ini demonstrasi metodologi.
    "2_revenue_genre": """
        SELECT a.Primary_Genre, a.Nama_Artis,
               COUNT(f.Order_ID) as Item_Terjual,
               ROUND(SUM(f.Gross_Rev), 2) as Total_Revenue,
               ROUND(AVG(f.Gross_Rev), 2) as Harga_Rata_Rata
        FROM Fact_Sales f JOIN Dim_Artis a ON f.Artis_SK = a.Artis_SK
        GROUP BY a.Primary_Genre, a.Nama_Artis ORDER BY Total_Revenue DESC;
    """,
    "3_bandcamp_friday": """
        SELECT w.Bandcamp_Friday,
               COUNT(DISTINCT w.Tanggal) as Jumlah_Hari,
               COUNT(f.Order_ID) as Total_Transaksi,
               ROUND(SUM(f.Gross_Rev), 2) as Total_Revenue,
               ROUND(SUM(f.Gross_Rev) / COUNT(DISTINCT w.Tanggal), 2) as Avg_Revenue_Per_Hari
        FROM Fact_Sales f JOIN Dim_Waktu w ON f.Time_ID = w.Time_ID
        GROUP BY w.Bandcamp_Friday;
    """,
    "4_negara_fisik": """
        SELECT l.Negara,
               COUNT(f.Order_ID) as Jumlah_Pembelian_Fisik,
               ROUND(SUM(f.Gross_Rev), 2) as Nilai_Transaksi_Fisik,
               ROUND(AVG(f.Gross_Rev), 2) as Avg_Spend_Per_Order
        FROM Fact_Sales f JOIN Dim_Lokasi l ON f.Lokasi_SK = l.Lokasi_SK
        JOIN Dim_Item i ON f.Item_SK = i.Item_SK WHERE i.Tipe_Item = 'Physical / Merch'
        GROUP BY l.Negara ORDER BY Nilai_Transaksi_Fisik DESC;
    """,
    "5_top_artis_merch": """
        SELECT a.Nama_Artis,
               COUNT(f.Order_ID) as Merch_Terjual,
               ROUND(SUM(f.Gross_Rev), 2) as Pendapatan_Kotor,
               ROUND(SUM(f.Bandcamp_Cut), 2) as Total_Potongan_Bandcamp,
               ROUND(SUM(f.Artist_Revenue), 2) as Pendapatan_Bersih_Artis
        FROM Fact_Sales f JOIN Dim_Artis a ON f.Artis_SK = a.Artis_SK
        JOIN Dim_Item i ON f.Item_SK = i.Item_SK WHERE i.Tipe_Item = 'Physical / Merch'
        GROUP BY a.Nama_Artis ORDER BY Pendapatan_Bersih_Artis DESC LIMIT 10;
    """,
    "6_tipe_vs_negara": """
        SELECT l.Negara, w.Kuartal, i.Tipe_Item,
               COUNT(f.Order_ID) as Volume_Penjualan,
               ROUND(SUM(f.Gross_Rev), 2) as Total_Revenue
        FROM Fact_Sales f JOIN Dim_Lokasi l ON f.Lokasi_SK = l.Lokasi_SK
        JOIN Dim_Waktu w ON f.Time_ID = w.Time_ID JOIN Dim_Item i ON f.Item_SK = i.Item_SK
        GROUP BY l.Negara, w.Kuartal, i.Tipe_Item ORDER BY Total_Revenue DESC;
    """,
    # CATATAN: Fan_Status & User = data SINTETIS, query ini demonstrasi metodologi.
    "7_subscriber_vs_standard": """
        SELECT p.Fan_Status, i.Tipe_Item,
               COUNT(DISTINCT p.Penggemar_SK) as Jumlah_Unik_Fans,
               COUNT(f.Order_ID) as Total_Transaksi,
               ROUND(SUM(f.Gross_Rev), 2) as Total_Belanja,
               ROUND(SUM(f.Gross_Rev) / COUNT(DISTINCT p.Penggemar_SK), 2) as Avg_Spend_Per_Fan
        FROM Fact_Sales f JOIN Dim_Penggemar p ON f.Penggemar_SK = p.Penggemar_SK
        JOIN Dim_Item i ON f.Item_SK = i.Item_SK
        GROUP BY p.Fan_Status, i.Tipe_Item;
    """
}

# Mengeksekusi dan mengekspor hasil ke CSV
for filename, sql_query in queries.items():
    df_olap = pd.read_sql_query(sql_query, conn)
    df_olap.to_csv(f'olap/hasil_query{filename}.csv', index=False)

conn.close()
print("🚀 Tahap 5 Selesai: 7 Output Analisis Bisnis berhasil diekstrak ke folder olap!")

🚀 Tahap 5 Selesai: 7 Output Analisis Bisnis berhasil diekstrak ke folder olap!
